**Pedido 1: Sequências de crescimento da receita**

Pergunta de negócio: “Quais foram os períodos mais longos em que a receita só subiu, mês após mês?”

Quero ver as 3 maiores sequências de toda a série. O critério de desempate é o maior crescimento acumulado.

In [0]:
WITH mensal AS (
    SELECT
        DATE_FORMAT(s.order_date, 'yyyy-MM') AS mes,
        ROUND(SUM(o.net_sales), 2) AS receita_liquida

    FROM workspace.`e-commerce_sales`.sales s
    JOIN workspace.`e-commerce_sales`.order_items o
      ON s.order_id = o.order_id

    GROUP BY DATE_FORMAT(s.order_date, 'yyyy-MM')
),

variacao AS (
    SELECT
        *,
        CASE WHEN receita_liquida > LAG(receita_liquida) OVER (ORDER BY mes) THEN 'subiu' ELSE 'caiu' END AS variacao
    FROM mensal
),

grupos AS(
    SELECT
        mes,
        receita_liquida,
        variacao,
        ROW_NUMBER() OVER (ORDER BY mes)
            - RANK() OVER (PARTITION BY variacao ORDER BY mes) AS grupo
    FROM variacao
),

sequencia AS(
    SELECT
        mes,
        receita_liquida,
        variacao,
        grupo,
        row_number() OVER(
            PARTITION BY variacao, grupo
            ORDER BY mes
        ) AS sequencia

    FROM grupos
),

ranking AS(
    SELECT
        grupo,
        variacao,
        MAX(sequencia) AS sequencia,
        ROUND(SUM(receita_liquida)) AS receita,
        dense_rank() OVER(
            ORDER BY MAX(sequencia) DESC, SUM(receita_liquida) DESC
        ) AS rank

    FROM sequencia
    GROUP BY grupo, variacao
    HAVING variacao = 'subiu'
),

top_3 AS(
    SELECT *
    FROM sequencia

    WHERE grupo IN(
        SELECT
            grupo
        FROM ranking
        WHERE rank <= 3
    )

    ORDER BY grupo, sequencia
)

SELECT
    MIN(mes) AS Mes_Inicial,
    MAX(mes) AS Mes_Final,
    MAX(sequencia) AS Qtd_Meses,
    ROUND(SUM(receita_liquida), 2) AS Total_Liquido

FROM top_3

GROUP BY grupo
ORDER BY Qtd_Meses DESC, Total_Liquido DESC 